**Crime and Enrichment Data Pipeline**

Package Imports

In [ ]:
import pandas as pd
import os

Ingestion Layer (Crime Data)

In [ ]:
basePath = '5YearSpanDatasets'

# List of forces that are going to be analysed
forces = ['metropolitan', 'thames', 'surrey', 'sussex']

# Array which is cumulatively appended for each force's crime data for each month between January 2019 to December 2024
allData = []

# Checks each folder in the updatedDatasets folder
for folder in os.listdir(basePath):
    # Concatenates the folder name to the updatedDatasets to create a new file path
    filePath = os.path.join(basePath, folder)
    # Checks if the folder actually exists
    if os.path.isdir(filePath):
        # Checks each file in the file path
        for file in os.listdir(filePath):
            # Checks the file contains one of the police forces in the filename
            if any(force in file.lower() for force in forces):
                # Updates file path to include the folder
               file_path = os.path.join(filePath, file)
               
               df=pd.read_csv(file_path)
               df['Month'] = folder
               allData.append(df)
    
# If statement to check the ingestion layer correctly loaded data
if allData:
    masterDf = pd.concat(allData, ignore_index=True)
else:
    print("No data found to process")


Initial Data Inspection

In [88]:
# Shows number of records in the master data
print(f"Before preprocessing, there are {len(masterDf)} values in the data")

# Shows each column and if they have any null values
print(f"\nList of null values in every column \n {masterDf.isnull().sum()}\n")

# Shows the unique values for each column
for col in masterDf:
    print(f" - The unique values for {col} are {masterDf[col].unique()}")


Before preprocessing, there are 9453125 values in the data

List of null values in every column 
 Crime ID                 2111038
Month                          0
Reported by                    0
Falls within                   0
Longitude                 176614
Latitude                  176614
Location                       0
LSOA code                 176644
LSOA name                 176644
Crime type                     0
Last outcome category    2111038
Context                  9453125
dtype: int64

 - The unique values for Crime ID are ['a8977a2a4e14252420371eb993d52e4d0b8288a7c833e66ba79592b3e2b982e7' nan
 '934e173f2bc2e1dd3a257b37939d8f97575d3eeb89ff0c3eee2c73f16630edc5' ...
 'cb80080e0164fa9d12767bcfe912ed902b1bc3db0ce83d50b7f6bc055d313392'
 'c49ac623430308ffb6f815e12725a9d3244c7bf3275592a22953ed4f5964c326'
 '14fb64f2b5de925c7c1cec2234f64d221dada2af05b6bc07fe41db9139e021ea']
 - The unique values for Month are ['2019-01' '2019-02' '2019-03' '2019-04' '2019-05' '2019-06' '2019-07'

Deduplication

In [ ]:
# Checks the number of total records in the crime data before any transformation or cleaning
totalBeforeDupRemoval = len(masterDf)
print(f"There are {totalBeforeDupRemoval} values in the dataset before any processing at all")

# Checks the number of duplicate records in the crime data
duplicateCount = masterDf.duplicated().sum()
print(f"There are {duplicateCount} duplicate rows.")

# Drops the duplicate records
masterDf = masterDf.drop_duplicates()

#
totalAfterDupRemoval = len(masterDf)
print(f"There are {totalAfterDupRemoval} values in the dataset after removing duplicates")

There are 9453125 values in the dataset before any processing at all
There are 1011718 duplicate rows.
There are 8441407 values in the dataset after removing duplicates


Null Value Handling

In [ ]:
# From cell 4 there are missing values in Crime ID, Longitude, Latitude, LSOA Code, LSOA Name, Last outcome category and Context

# Shows each column and if they have any null values
print(f"\nList of null values in every column before handling null values\n {masterDf.isnull().sum()}\n")

# In the data, Crime ID, Longitude, Latitude, LSOA Code, LSOA Name, Last outcome category and Context contained null values

# Context was empty for every single record so can be dropped
masterDf = masterDf.drop("Context", axis='columns')

# For remaining columns, too much crime data would be lost if dropped so placeholders were assigned instead

# Crime ID: add placeholder value "No Crime ID"
# Longitude and Latitude: add placeholder value 0
# LSOA code: add placeholder value "No LSOA Code Assigned"
# LSOA name: add placeholder value "No LSOA name Assigned"
# Last outcome category: add placeholder value "Unknown outcome category"
# Context: drop since every record has null values in context

placeholders = {
    'Crime ID': 'No Crime ID',
    'Longitude': 0,
    'Latitude' : 0,
    'LSOA code' : 'No LSOA Code assigned',
    'LSOA name' : 'No LSOA Name assigned',
    'Last outcome category' : 'Unknown outcome category'
}

masterDf = masterDf.fillna(value=placeholders)

# Shows each column and if they have any null values
print(f"\nList of null values in every column \n {masterDf.isnull().sum()}\n")

There are 8441407 total records

List of null values in every column before handling null values
 Crime ID                 1155292
Month                          0
Reported by                    0
Falls within                   0
Longitude                 167326
Latitude                  167326
Location                       0
LSOA code                 167356
LSOA name                 167356
Crime type                     0
Last outcome category    1155292
Context                  8441407
dtype: int64


List of null values in every column 
 Crime ID                 0
Month                    0
Reported by              0
Falls within             0
Longitude                0
Latitude                 0
Location                 0
LSOA code                0
LSOA name                0
Crime type               0
Last outcome category    0
dtype: int64



Column Handling

In [ ]:
# Dropping "Falls within" column as each record mirrored the value within the "Reported by" column
masterDf = masterDf.drop("Falls within", axis='columns')

Crime ID
Month
Reported by
Longitude
Latitude
Location
LSOA code
LSOA name
Crime type
Last outcome category


Text Standardisation

In [ ]:
# Clean Force names to be just the name of the police force without "Force" or other keywords
masterDf['Force'] = masterDf['Reported by'].str.replace("Police", "").str.strip()

# Clean metropolitan
masterDf.loc[masterDf['Force'].str.contains('Metropolitan', case=False), 'Force'] = 'Metropolitan'
masterDf['Force'] = masterDf['Force'].str.title()

# Standardize 'Crime type' and 'Last outcome category'
masterDf['Crime type'] = masterDf['Crime type'].str.strip().str.capitalize()
masterDf['Last outcome category'] = masterDf['Last outcome category'].str.strip().str.capitalize()

# Dropped Reported by column since a cleaner, standardised Force column was derived from it
masterDf = masterDf.drop('Reported by', axis='columns')

# For loop to ensure that the column was definitely dropped
for col in masterDf:
    print(col)
    
print(f"\n{masterDf.head()}")

0          Metropolitan Police Service
1          Metropolitan Police Service
2          Metropolitan Police Service
3          Metropolitan Police Service
4          Metropolitan Police Service
                      ...             
9453120           Thames Valley Police
9453121           Thames Valley Police
9453122           Thames Valley Police
9453123           Thames Valley Police
9453124           Thames Valley Police
Name: Reported by, Length: 8441407, dtype: object
Crime ID
Month
Longitude
Latitude
Location
LSOA code
LSOA name
Crime type
Last outcome category
Force


Year Conversion 

In [ ]:
# Converting the existing time data into a readable pandas time attribute for future analysis
yearConversion = pd.to_datetime(masterDf['Month'])
masterDf['Year'] = yearConversion.dt.year

,Crime ID,Month,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Force,Year
0,a8977a2a4e14252420371eb993d52e4d0b8288a7c833e6...,2019-01,-0.709911,50.784615,On or near Rochester Way,E01031384,Arun 018E,Violence and sexual offences,Status update unavailable,Metropolitan,2019
1,No Crime ID,2019-01,0.140192,51.582311,On or near Hatch Grove,E01000027,Barking and Dagenham 001A,Anti-social behaviour,Unknown outcome category,Metropolitan,2019
2,934e173f2bc2e1dd3a257b37939d8f97575d3eeb89ff0c...,2019-01,0.140192,51.582311,On or near Hatch Grove,E01000027,Barking and Dagenham 001A,Burglary,Status update unavailable,Metropolitan,2019
3,4f5b7e424bc78b1fb8c32e07da61176d2cbc5a3849d8e1...,2019-01,0.140634,51.583427,On or near Rams Grove,E01000027,Barking and Dagenham 001A,Burglary,Status update unavailable,Metropolitan,2019
4,53d960600a4a9f54b785f598af4c75bdef2f79bce1a41b...,2019-01,0.141143,51.590873,On or near Furze Farm Close,E01000027,Barking and Dagenham 001A,Drugs,Investigation complete; no suspect identified,Metropolitan,2019
...,...,...,...,...,...,...,...,...,...,...,...
9453120,1a290c6bed646acafa5ae267878c9e74ad363ab04a4de3...,2024-12,0.000000,0.000000,No Location,No LSOA Code assigned,No LSOA Name assigned,Other crime,Under investigation,Thames Valley,2024
9453121,b66a5c6320476a6a6fae6f3d8600e7a9c39a780cee7874...,2024-12,0.000000,0.000000,No Location,No LSOA Code assigned,No LSOA Name assigned,Other crime,Under investigation,Thames Valley,2024
9453122,cb80080e0164fa9d12767bcfe912ed902b1bc3db0ce83d...,2024-12,0.000000,0.000000,No Location,No LSOA Code assigned,No LSOA Name assigned,Other crime,Under investigation,Thames Valley,2024
9453123,c49ac623430308ffb6f815e12725a9d3244c7bf3275592...,2024-12,0.000000,0.000000,No Location,No LSOA Code assigned,No LSOA Name assigned,Other crime,Awaiting court outcome,Thames Valley,2024


Crime Data Aggregation

In [ ]:
# Creating the aggregations, grouping by the defined grain (Force X Year X LSOA name x Crime Type)
crimeAggregation = masterDf.groupby(['Force', 'Year', 'LSOA name', 'Crime type']).agg(
    crimeCount=('Crime ID', 'count'),
    Latitude=('Latitude', 'mean'),
    Longitude=('Longitude', 'mean')
).reset_index()

crimeAggregation = crimeAggregation.rename(columns={'crimeCount': 'Crime count'})

crimeAggregation.head(10)

,Force,Year,LSOA name,Crime type,Crime count,Latitude,Longitude
0,Metropolitan,2019,Adur 006A,Other theft,1,50.837110,-0.335092
1,Metropolitan,2019,Amber Valley 016A,Violence and sexual offences,1,52.982886,-1.493708
2,Metropolitan,2019,Arun 009F,Violence and sexual offences,1,50.816571,-0.543651
3,Metropolitan,2019,Arun 017A,Other crime,1,50.785967,-0.662250
4,Metropolitan,2019,Arun 017A,Violence and sexual offences,1,50.785967,-0.662250
5,Metropolitan,2019,Arun 018E,Violence and sexual offences,2,50.782778,-0.711252
6,Metropolitan,2019,Ashfield 004D,Violence and sexual offences,1,53.127231,-1.256562
7,Metropolitan,2019,Ashfield 006E,Violence and sexual offences,2,53.117919,-1.283124
8,Metropolitan,2019,Ashford 002A,Vehicle crime,1,51.217299,0.829627
9,Metropolitan,2019,Ashford 004I,Violence and sexual offences,1,51.157089,0.855962


Ingestion Layer (ONS Population Data)

In [ ]:
xls = pd.ExcelFile('policeforceareas1991to2024.xlsx')
# Extracting only the Mid-2011 to Mid-2020 and Mid-2021 to Mid-2024 sheet as these contain the years that are to be analysed
pop_11_20 = pd.read_excel(xls, 'Mid-2011 to Mid-2020', skiprows=3)
pop_21_24 = pd.read_excel(xls, 'Mid-2021 to Mid-2024', skiprows=3)

# Concatinating the data together to form one raw data set of the ONS Population
popAll = pd.concat([pop_11_20, pop_21_24], axis=0)
popAll.head(12)




,PFA 2023 Code,PFA 2023 Name,Year,F0,F1,F2,F3,F4,F5,F6,...,M76,M77,M78,M79,M80,M81,M82,M83,M84,M85
0,E23000001,Metropolitan Police,2011,61350,59379,57697,57186,54527,52518,49086,...,16821,15777,14827,14280,13328,12373,10812,9522,8565,40890
1,E23000001,Metropolitan Police,2012,64693,60570,58582,57211,56859,54303,52219,...,17456,16016,15024,14039,13417,12387,11448,9909,8637,42292
2,E23000001,Metropolitan Police,2013,63201,63403,59518,57938,56781,56590,53985,...,17250,16625,15262,14225,13230,12591,11369,10485,9005,43356
3,E23000001,Metropolitan Police,2014,61334,61617,61625,58383,57221,56382,56211,...,17521,16447,15850,14445,13404,12422,11739,10384,9581,45254
4,E23000001,Metropolitan Police,2015,61691,59485,59715,60000,57438,56610,55980,...,17556,16738,15633,14966,13568,12501,11507,10822,9375,46343
5,E23000001,Metropolitan Police,2016,62053,59758,57636,58046,58540,56678,55992,...,17418,16780,15894,14760,14044,12699,11599,10660,9883,47630
6,E23000001,Metropolitan Police,2017,61005,59802,57395,55825,56215,57172,55737,...,15718,16590,15989,15071,13925,13076,11839,10749,9760,49023
7,E23000001,Metropolitan Police,2018,58483,58615,57530,55457,54113,55066,56114,...,16489,15000,15801,15157,14186,13094,12192,10941,9873,49872
8,E23000001,Metropolitan Police,2019,56737,56173,56138,55548,53807,52871,54087,...,17674,15818,14294,15043,14337,13362,12218,11271,10072,51213
9,E23000001,Metropolitan Police,2020,55300,54589,53733,54089,53768,52479,51888,...,18547,16808,14974,13353,14029,13214,12327,11166,10156,50855


In [ ]:
# Calculates the total population for a force for a given year
populationColumns = popAll.columns[3:]
popAll['Population'] = popAll[populationColumns].sum(axis=1)

popClean = popAll[['PFA 2023 Name', 'Year', 'Population']].copy()
popClean.columns = ['Force', 'Year', 'Population']
popClean = popClean.reset_index(drop=True)

print(popClean.head())

                 Force  Year  Population
0  Metropolitan Police  2011     8196995
1  Metropolitan Police  2012     8313253
2  Metropolitan Police  2013     8431450
3  Metropolitan Police  2014     8539625
4  Metropolitan Police  2015     8651877


In [98]:
popClean['Force'] = popClean['Force'].str.replace("Police", "").str.strip()

print(popClean.head(12))


           Force  Year  Population
0   Metropolitan  2011     8196995
1   Metropolitan  2012     8313253
2   Metropolitan  2013     8431450
3   Metropolitan  2014     8539625
4   Metropolitan  2015     8651877
5   Metropolitan  2016     8735914
6   Metropolitan  2017     8768352
7   Metropolitan  2018     8825170
8   Metropolitan  2019     8880978
9   Metropolitan  2020     8858677
10       Cumbria  2011      499817
11       Cumbria  2012      498835


In [99]:
forceYearTotals = masterDf.groupby(['Force', 'Year']).size().reset_index(name='Total Force Crime Count')
forceRatesBaseline = pd.merge(forceYearTotals, popClean, on=['Force', 'Year'], how='left')
forceRatesBaseline['Force Crimes per 1000'] = (forceRatesBaseline['Total Force Crime Count'] / forceRatesBaseline['Population']) * 1000

In [100]:
force_lookup = {
    # Metropolitan 
    'Barking and Dagenham': 'Metropolitan', 'Barnet': 'Metropolitan', 'Bexley': 'Metropolitan',
    'Brent': 'Metropolitan', 'Bromley': 'Metropolitan', 'Camden': 'Metropolitan',
    'Croydon': 'Metropolitan', 'Ealing': 'Metropolitan', 'Enfield': 'Metropolitan',
    'Greenwich': 'Metropolitan', 'Hackney': 'Metropolitan', 'Hammersmith and Fulham': 'Metropolitan',
    'Haringey': 'Metropolitan', 'Harrow': 'Metropolitan', 'Havering': 'Metropolitan',
    'Hillingdon': 'Metropolitan', 'Hounslow': 'Metropolitan', 'Islington': 'Metropolitan',
    'Kensington and Chelsea': 'Metropolitan', 'Kingston upon Thames': 'Metropolitan',
    'Lambeth': 'Metropolitan', 'Lewisham': 'Metropolitan', 'Merton': 'Metropolitan',
    'Newham': 'Metropolitan', 'Redbridge': 'Metropolitan', 'Richmond upon Thames': 'Metropolitan',
    'Southwark': 'Metropolitan', 'Sutton': 'Metropolitan', 'Tower Hamlets': 'Metropolitan',
    'Waltham Forest': 'Metropolitan', 'Wandsworth': 'Metropolitan', 'Westminster': 'Metropolitan',
    'City of London': 'Metropolitan', 
    # Surrey
    'Elmbridge': 'Surrey', 'Epsom and Ewell': 'Surrey', 'Guildford': 'Surrey',
    'Mole Valley': 'Surrey', 'Reigate and Banstead': 'Surrey', 'Runnymede': 'Surrey',
    'Spelthorne': 'Surrey', 'Surrey Heath': 'Surrey', 'Tandridge': 'Surrey',
    'Waverley': 'Surrey', 'Woking': 'Surrey',

    # Sussex
    'Adur': 'Sussex', 'Arun': 'Sussex', 'Brighton and Hove': 'Sussex',
    'Chichester': 'Sussex', 'Crawley': 'Sussex', 'Eastbourne': 'Sussex',
    'Hastings': 'Sussex', 'Horsham': 'Sussex', 'Lewes': 'Sussex',
    'Mid Sussex': 'Sussex', 'Rother': 'Sussex', 'Wealden': 'Sussex', 'Worthing': 'Sussex',

    # Thames Valley
    'Bracknell Forest': 'Thames Valley', 'Buckinghamshire': 'Thames Valley', 'Cherwell': 'Thames Valley',
    'Milton Keynes': 'Thames Valley', 'Oxford': 'Thames Valley', 'Reading': 'Thames Valley',
    'Slough': 'Thames Valley', 'South Oxfordshire': 'Thames Valley', 'Vale of White Horse': 'Thames Valley',
    'West Berkshire': 'Thames Valley', 'West Oxfordshire': 'Thames Valley',
    'Windsor and Maidenhead': 'Thames Valley', 'Wokingham': 'Thames Valley'
}

#add column for lsoa name

iod2025 = pd.read_csv("File_7_IoD2025_All_Ranks_Scores_Deciles_Population_Denominators.csv")
iod2025['Force'] = iod2025['Local Authority District name (2024)'].map(force_lookup)

excludedEntries = iod2025['Force'].isna().sum()
print(f"There are {excludedEntries} out of {len(iod2025)} LSOAs that are outside the 4 forces")

There are 25528 out of 33755 LSOAs that are outside the 4 forces


In [101]:
iodFiltered = iod2025.dropna(subset=['Force']).copy()
iodFiltered['is_decile_1'] = iodFiltered['Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)'] == 1

# display(iodFiltered)
print(iodFiltered['is_decile_1'].value_counts())

imd_final = iodFiltered.groupby('Force').agg(
    AvgIMDScore=('Index of Multiple Deprivation (IMD) Score', 'mean'),
    AvgIncomeDecile=('Income Decile (where 1 is most deprived 10% of LSOAs)', 'mean'),
    NeighbourhoodCount=('LSOA code (2021)', 'count'),
    HighlyDeprivedCount=('is_decile_1', 'sum')
    
).reset_index()

imd_final['HighlyDeprivedPercentage'] = (imd_final['HighlyDeprivedCount'] / imd_final['NeighbourhoodCount']) * 100

# 4. Final check - you should have exactly 4 rows now
display(imd_final)

is_decile_1
False    7968
True      259
Name: count, dtype: int64


,Force,AvgIMDScore,AvgIncomeDecile,NeighbourhoodCount,HighlyDeprivedCount,HighlyDeprivedPercentage
0,Metropolitan,22.128675,4.543052,4994,201,4.024830
1,Surrey,10.108495,7.528512,719,0,0.000000
2,Sussex,17.677184,6.051506,1029,48,4.664723
3,Thames Valley,12.673253,6.952189,1485,10,0.673401


In [102]:
finalData = pd.merge(crimeAggregation, forceRatesBaseline, on=['Force', 'Year'], how='left')
completeData = pd.merge(finalData, imd_final, on=['Force'], how='left')

display(completeData)

,Force,Year,LSOA name,Crime type,Crime count,Latitude,Longitude,Total Force Crime Count,Population,Force Crimes per 1000,AvgIMDScore,AvgIncomeDecile,NeighbourhoodCount,HighlyDeprivedCount,HighlyDeprivedPercentage
0,Metropolitan,2019,Adur 006A,Other theft,1,50.837110,-0.335092,980566,8880978,110.411939,22.128675,4.543052,4994,201,4.024830
1,Metropolitan,2019,Amber Valley 016A,Violence and sexual offences,1,52.982886,-1.493708,980566,8880978,110.411939,22.128675,4.543052,4994,201,4.024830
2,Metropolitan,2019,Arun 009F,Violence and sexual offences,1,50.816571,-0.543651,980566,8880978,110.411939,22.128675,4.543052,4994,201,4.024830
3,Metropolitan,2019,Arun 017A,Other crime,1,50.785967,-0.662250,980566,8880978,110.411939,22.128675,4.543052,4994,201,4.024830
4,Metropolitan,2019,Arun 017A,Violence and sexual offences,1,50.785967,-0.662250,980566,8880978,110.411939,22.128675,4.543052,4994,201,4.024830
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
580907,Thames Valley,2024,Wokingham 020E,Other crime,2,51.374259,-0.802241,194795,2640201,73.780367,12.673253,6.952189,1485,10,0.673401
580908,Thames Valley,2024,Wokingham 020E,Other theft,1,51.375766,-0.804458,194795,2640201,73.780367,12.673253,6.952189,1485,10,0.673401
580909,Thames Valley,2024,Wokingham 020E,Public order,1,51.375957,-0.800300,194795,2640201,73.780367,12.673253,6.952189,1485,10,0.673401
580910,Thames Valley,2024,Wokingham 020E,Vehicle crime,3,51.371635,-0.800375,194795,2640201,73.780367,12.673253,6.952189,1485,10,0.673401


In [ ]:
grainDuplicates = completeData.duplicated(subset=['Force', 'Year', 'LSOA name', 'Crime type']).sum()
print(f"Validation - Duplicate records at reporting grain: {grainDuplicates}")

print("\nFinal Missing Value Summary:")
print(completeData[['Force', 'Year', 'LSOA name', 'Crime type']].isnull().sum())

# 3. High-level Statistics Summary for Hand-off Verification
print(f"\n--- Final Dataset Summary Statistics ---")
print(f"Total Grain Rows: {len(completeData)}")
print(f"Unique Police Forces Covered: {completeData['Force'].nunique()} ({completeData['Force'].unique()})")
print(f"Temporal Window Span: {completeData['Year'].min()} to {completeData['Year'].max()}")

# 4. Save Final Asset
completeData.to_csv('AnotherFinal_Integrated_Crime_Social_Analytics_Pipeline.csv', index=False)
print("\nSuccess: BI-ready reporting dataset successfully exported.")





Validation - Duplicate records at reporting grain: 0

Final Missing Value Summary:
Force         0
Year          0
LSOA name     0
Crime type    0
dtype: int64

--- Final Dataset Summary Statistics ---
Total Grain Rows: 580912
Unique Police Forces Covered: 4 (['Metropolitan' 'Surrey' 'Sussex' 'Thames Valley'])
Temporal Window Span: 2019 to 2024

Success: BI-ready reporting dataset successfully exported.
